# Multi-Model Operating Points & Athena-Matched Comparison

Loads trained models from a list of `results/` folders, selects the best trained model per kinematic region ($E_T \times |\eta|$), and evaluates $P_D$, $SP$, and $P_F$ for:
1. **Training Best SP** (validation-selected operating point)
2. **Calibrated Hard Cut** (matched to full-dataset Athena $P_D$)
3. **Linear Pileup-Corrected Cut $y_{\text{cut}}(\langle\mu\rangle)$** (matched to full-dataset Athena $P_D$)
4. **Athena Cut Baseline** (`loose`, `medium`, or `tight`)

Includes a `config_name` column in the tidy DataFrame so models can be filtered and compared directly.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.core.Validation.RegionalOperatingPointsConsolidator import (
    MultiModelAthenaMatchedConsolidator,
)

print(f"Project root resolved: {PROJECT_ROOT}")

### 1. Configuration & Results Folders
Specify the list of results directories (`config_name<Name>_id<Timestamp>`). Configurations are auto-resolved from `config/NeuralRinger/`.

In [ ]:
CONFIG_DIR = PROJECT_ROOT / "config/NeuralRinger"
RESULTS_FOLDERS = [
    PROJECT_ROOT / "results/config_nameModelV9_HighBatch_100Rings_id20261105222012",
    PROJECT_ROOT / "results/config_nameModelV9_HighBatch_50Rings_id20261109122259",
    PROJECT_ROOT / "results/config_nameModelV10_HighBatch_100Rings_id20261112210846",
    PROJECT_ROOT / "results/config_nameModelV10_HighBatch_50Rings_id20261115051346",
]
MANIFEST_PATH = PROJECT_ROOT / "results/split_manifest.json"

WORKING_POINT = "loose"
BINS_MU = 10
N_BOOTSTRAPS_PILEUP = 100
EFFICIENCIES_CSV = None
INCLUDE_TRAINING_BEST_SP = True

AS_PERCENTAGE = True
PRECISION = 2
DECIMAL_COMMA = False
MATH_DELIMITERS = True
BOLD_BEST_SP = True

OUTPUT_DIR = PROJECT_ROOT / "notebooks/Tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_OUTPUT = OUTPUT_DIR / "multi_model_operating_points_tidy.parquet"
GRID_PARQUET_OUTPUT = OUTPUT_DIR / "multi_model_operating_points_grid.parquet"
LATEX_OUTPUT = OUTPUT_DIR / "multi_model_operating_points.tex"

for folder in RESULTS_FOLDERS:
    print(f"Results folder: {folder} (exists={folder.exists()})")

### 2. Evaluate All Models Across Regions
Instantiate `MultiModelAthenaMatchedConsolidator` from `RESULTS_FOLDERS` and run evaluation.

In [ ]:
consolidator = MultiModelAthenaMatchedConsolidator.from_results_dirs(
    results_dirs=RESULTS_FOLDERS,
    config_dir=CONFIG_DIR,
    manifest_path=MANIFEST_PATH if MANIFEST_PATH.exists() else None,
    working_point=WORKING_POINT,
    bins_mu=BINS_MU,
    n_bootstraps_pileup=N_BOOTSTRAPS_PILEUP,
    efficiencies_csv=EFFICIENCIES_CSV,
    include_training_best_sp=INCLUDE_TRAINING_BEST_SP,
)

records = consolidator.scan_and_evaluate()
print(f"Total extracted records: {len(records)}")

### 3. Tidy DataFrame & Model Selection by `config_name`
Inspect the tidy metrics table and filter by `config_name` to compare specific models.

In [ ]:
tidy_df = consolidator.build_tidy_dataframe()
print("Available config_name values:", tidy_df["config_name"].unique().tolist())
display(tidy_df[["iet", "ieta", "config_name", "method", "pd", "sp", "pf"]].head(15))

In [ ]:
# Example: select specific models (plus Athena baseline) by config_name
SELECTED_CONFIGS = [
    "Athena",
    "ModelV9_HighBatch_100Rings",
    "ModelV10_HighBatch_100Rings",
]
filtered_df = tidy_df[tidy_df["config_name"].isin(SELECTED_CONFIGS)]
display(filtered_df[["iet", "ieta", "config_name", "method", "pd", "sp", "pf"]].head(15))

### 4. 2D Kinematic Grid Comparison Table
Build and display the formatted $E_T \times |\eta|$ comparison grid across all loaded models.

In [ ]:
grid_table = consolidator.build_grid_table(
    as_percentage=AS_PERCENTAGE,
    precision=PRECISION,
    decimal_comma=DECIMAL_COMMA,
    math_delimiters=MATH_DELIMITERS,
    bold_best_sp=BOLD_BEST_SP,
)
display(grid_table)

### 5. Export Parquet & LaTeX Tables
Save the tidy DataFrame (with `config_name`), the 2D grid table, and the publication-ready LaTeX snippet.

In [ ]:
saved_tidy = consolidator.export_parquet(PARQUET_OUTPUT, df=tidy_df)
saved_grid = consolidator.export_parquet(GRID_PARQUET_OUTPUT, df=grid_table)

latex_code = consolidator.to_latex_snippet(
    grid_table=grid_table,
    caption=(
        rf"Comparison of Athena {WORKING_POINT.capitalize()} Cut, Training Best SP, "
        rf"Calibrated Hard Cut, and Linear Pileup-Corrected Cut $y_\mathrm{{cut}}(\langle\mu\rangle)$ "
        rf"across $E_T$ and $|\eta|$ regions."
    ),
    label="tab:multi_model_operating_points_comparison",
    resize_box=True,
    as_percentage=AS_PERCENTAGE,
    precision=PRECISION,
    decimal_comma=DECIMAL_COMMA,
)
saved_latex = consolidator.export_latex(LATEX_OUTPUT, latex_code=latex_code)

print(f"Tidy Parquet saved to: {saved_tidy}")
print(f"Grid Parquet saved to: {saved_grid}")
print(f"LaTeX saved to:        {saved_latex}")

### 6. Mean Performance Summary Across Regions ($P_D$, $SP$, $P_F$)
Computes the mean (and standard deviation across kinematic regions) of $P_D$, $SP$, and $P_F$ for each `config_name` and operating point method.

In [ ]:
scale = 100.0 if AS_PERCENTAGE else 1.0
unit = "[%]" if AS_PERCENTAGE else ""

summary_df = (
    tidy_df.groupby(["config_name", "method"], sort=False)[["pd", "sp", "pf"]]
    .agg(["mean", "std"])
    .mul(scale)
    .round(PRECISION)
)
summary_df.columns = [
    f"{metric.upper()} {stat} {unit}".strip()
    for metric, stat in summary_df.columns
]
summary_df = summary_df.reset_index()
display(summary_df)

summary_parquet = consolidator.export_parquet(
    OUTPUT_DIR / "multi_model_mean_summary.parquet",
    df=summary_df,
)
print(f"Mean summary Parquet saved to: {summary_parquet}")